In [15]:
import geopandas as gpd
import pandas as pd 
import sys
import os
import requests
import rasterio
from shapely.geometry import box
import numpy as np
from osgeo import gdal
import zipfile
import math


origin = '/workspace/'
sys.path.append('/media/')

In [2]:
from FieldWaterUseTools.FuncBox.Misc import getFilelist, path_safe


year = 2023
master = f"{origin}fields/04_Predictions/GERMANY/FromScratch_IACS_dilate_True_BorderEdgeCutted_RGB_NDVI_exclude_True_with_overlap_47/{year}/"
chips_folder = f"{master}chips_folder/unmasked_chips/"
masked_folder = path_safe(f"{master}chips_folder/masked_chips/")
files = getFilelist(chips_folder, '.tif')
ds = gdal.Open(f"{master}vrt/Masked_THUENEN_CTM_2023.tif")

In [4]:
conti = []

for file in files:
    # load masked stack
    chip = ds.GetRasterBand(1).ReadAsArray(
        xoff=int(file.split('X_')[-1].split('_')[0]),
        yoff=int(file.split('X_')[-1].split('_')[2].split('.')[0]),
        win_xsize=236,  
        win_ysize=236 
    )

    if np.nansum(chip) > 0:

        with rasterio.open(file) as src:
            bounds = src.bounds  # left, bottom, right, top
            geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
            crs = src.crs

            conti.append({
                "filename": os.path.basename(file),
                "geometry": geom,
                "crs_used": str(crs)
            })

            chip2 = ds.GetRasterBand(2).ReadAsArray(
                xoff=int(file.split('X_')[-1].split('_')[0]),
                yoff=int(file.split('X_')[-1].split('_')[2].split('.')[0]),
                win_xsize=236,  
                win_ysize=236 
            )

            chip3 = ds.GetRasterBand(3).ReadAsArray(
                xoff=int(file.split('X_')[-1].split('_')[0]),
                yoff=int(file.split('X_')[-1].split('_')[2].split('.')[0]),
                win_xsize=236,  
                win_ysize=236 
            )

            stack = np.stack([chip, chip2, chip3], axis=0)

            with rasterio.open(
                f"{masked_folder}chips_masked256{file.split('_unmasked256')[-1]}",
                "w",
                driver="GTiff",
                height=stack.shape[1],
                width=stack.shape[2],
                count=stack.shape[0],
                dtype=stack.dtype,
                crs=crs,          # z.B. von einem Referenz-Datensatz: ref_ds.crs
                transform=src.transform  # z.B. ref_ds.transform
            ) as dst:
                dst.write(stack)

In [5]:
gdf = gpd.GeoDataFrame(conti, geometry="geometry", crs=conti[0]["crs_used"])
gdf.to_file(f"{master}{year}_grid_tiles.gpkg", driver="GPKG", layer="tiles")

In [6]:
from FieldWaterUseTools.FuncBox.FieldFuncis import predicted_chips_to_vrt

predicted_chips_to_vrt(f"{master}chips_folder/", 'masked_chips', 256, 20, path_safe(f"{master}vrt/"), pyramids=True)

In [42]:
# zip em
mfiles = getFilelist(masked_folder, '.tif')
print(len(mfiles))
n = 8
chunk_size = math.ceil(len(mfiles) / n)
chunks = [mfiles[i:i + chunk_size] for i in range(0, len(mfiles), chunk_size)]

65509


In [43]:
len(chunks[0])

8189

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def zip_chunk(args):
    idx, chunk, masked_folder = args

    zip_name = os.path.join(masked_folder, f"masked_{idx}.zip")

    with zipfile.ZipFile(
        zip_name,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as zf:
        for tif_path in chunk:
            zf.write(tif_path, arcname=os.path.basename(tif_path))



with ProcessPoolExecutor(max_workers=30) as executor:
    executor.map(
        zip_chunk,
        [(idx, chunk, masked_folder) for idx, chunk in enumerate(chunks)]
    )